In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import bacco
from random import sample 
import copy

In [ ]:
import os
os.chdir("/lscratch/fgmaion/MTNG-resims/src")
import utils

In [ ]:
plt.rcParams["font.family"] = "serif"
plt.rcParams["mathtext.fontset"] = "dejavuserif"

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
rho_c = 2.7754e11
M_T = 500**3 * rho_c

In [ ]:
m_bins1 = np.arange(11,12.51, 0.05)
m_bins2 = np.arange(12.5,13.55, 0.1)
m_bins3 = np.arange(13.5,14.6, 0.5)

m_centers_1 = 0.5 * ( m_bins1[1:] + m_bins1[:-1] )
m_centers_2 = 0.5 * ( m_bins2[1:] + m_bins2[:-1] )
m_centers_3 = 0.5 * ( m_bins3[1:] + m_bins3[:-1] )

In [ ]:
frac = 4 * ( np.sum(10**(m_centers_1)) + np.sum(10**(m_centers_2)) + np.sum(10**(m_centers_3)) ) / M_T
print(frac)

In [ ]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [ ]:
sim = bacco.utils.load_MTNG(adr="/cosmos_storage/simulations/MTNG/", snap=264)

In [ ]:
sim.sub['parent_halo']['index']

In [ ]:
fig, ax = plt.subplots(dpi=150, figsize=(5.5,5))

ax.hist(np.log10(1e10*sim.fof['halo_mvir']), bins=np.linspace(11,15,100), log=True);

In [ ]:
Nbins = 30

splitter = utils.split_halos(sim)
#dic_full = splitter.stellar_mf(mass_edges=[8, 15], nbins=Nbins)

### Define ranges of halo mass

In [ ]:
mass_edges=np.array([[11,11.2],[11.2,11.4],[11.4,11.6],[11.6,11.8],[11.8,12],\
                     [12,12.2],[12.2,12.4],[12.4,12.6],[12.6,12.8],[12.8,13],\
                     [13,13.2],[13.2,13.4],[13.4,13.6],[13.6,13.8],[13.8,14],\
                     [14,14.2],[14.2,14.4],[14.4,14.6],[14.8,15]])

c_Nhalos = [64,100,100,100,100,\
            100,120,100,80,60,\
            30,20,10,6,4,\
            2,2,2,0]

c_Nhalos_10 = [2,0,0,2,0,\
            0,0,2,0,0,\
            2,0,2,0,0,\
            0,0,0,0]

c_Nhalos_100 = [6,10,10,10,10,\
            10,12,10,4,4,\
            4,2,2,2,2,\
            2,0,0,0]

# mass_edges=np.array([[13,13.2],[13.2,13.4],[13.8,14],\
#                      [14,14.2],[14.2,14.4],[14.8,15]])

### Get the subhalo selection

In [ ]:
sels=['2', 'Custom']

In [ ]:
sel_total = splitter.subhalo_sel(mhalo_edges=np.array([[8,15]]), vmax_sel=False, Nhalos=None)

In [ ]:
nbins=20
smf_total = splitter.stellar_mf(gal_sel=sel_total['gal_sel'], sel_mask=sel_total, nbins=nbins)

In [ ]:
sel = {}
smf = {}
smf['2'] = np.zeros((100,nbins-1))
smf['vmax_2'] = np.zeros((100,nbins-1))
smf['Custom'] = np.zeros((100,nbins-1))
smf['vmax_Custom'] = np.zeros((100,nbins-1))

for i in range(1):
    sel[i] = {}
    # sel[i]['2'] = splitter.subhalo_sel(mhalo_edges=mass_edges, vmax_sel=False, Nhalos=2)
    # sel[i]['vmax_2'] = splitter.subhalo_sel(mhalo_edges=mass_edges, vmax_sel=True, Nhalos=2)
    # sel[i]['Custom'] = splitter.subhalo_sel(mhalo_edges=mass_edges, vmax_sel=False, Nhalos=c_Nhalos)
    sel[i]['vmax_Custom'] = splitter.subhalo_sel(mhalo_edges=mass_edges, vmax_sel=True, Nhalos=c_Nhalos)

    # smf['2'][i,:] = splitter.stellar_mf(gal_sel=sel[i]['2']['gal_sel'], sel_mask=sel[i]['2'], nbins=nbins)['smf']
    # smf['vmax_2'][i,:] = splitter.stellar_mf(gal_sel=sel[i]['vmax_2']['gal_sel'], sel_mask=sel[i]['vmax_2'], nbins=nbins, vmax_sel=True)['smf']
    # smf['Custom'][i,:] = splitter.stellar_mf(gal_sel=sel[i]['Custom']['gal_sel'], sel_mask=sel[i]['Custom'], nbins=nbins)['smf']
    smf['vmax_Custom'][i,:] = splitter.stellar_mf(gal_sel=sel[i]['vmax_Custom']['gal_sel'], sel_mask=sel[i]['vmax_Custom'], nbins=nbins, vmax_sel=True)['smf']

In [ ]:
sel_vmax = splitter.subhalo_sel(mhalo_edges=mass_edges, vmax_sel=True, Nhalos=2)

Selecting 1000 Halos

In [ ]:
sel['Custom'] = splitter.subhalo_sel(mhalo_edges=mass_edges, vmax_sel=False, Nhalos=c_Nhalos)
sel['vmax_Custom'] = splitter.subhalo_sel(mhalo_edges=mass_edges, vmax_sel=True, Nhalos=c_Nhalos)

100 Halos

In [ ]:
sel_100 = {}
sel_100['Custom'] = splitter.subhalo_sel(mhalo_edges=mass_edges, vmax_sel=False, Nhalos=c_Nhalos_100)
sel_100['vmax_Custom'] = splitter.subhalo_sel(mhalo_edges=mass_edges, vmax_sel=True, Nhalos=c_Nhalos_100)

In [ ]:
def save_halo_sel(sel_mask, vmax_sel=False, Nhalos=None):

    idx = []
    if vmax_sel is True:
        for m in range(len(sel_mask['vmax_Custom']['h_idx'])):
            for v in range(len(sel_mask['vmax_Custom']['h_idx'][m])):
                idx.extend(sel_mask['vmax_Custom']['h_idx'][m][v])

        idx = np.unique(np.array(idx))

        np.save("/lscratch/fgmaion/MTNG-resims/halo_selections/{:d}/vmax_custom_selection".format(Nhalos), [idx])
    else:
        idx = sel_mask['Custom']['h_idx']

        np.save("/lscratch/fgmaion/MTNG-resims/halo_selections/{:d}/regular_selection".format(Nhalos), [idx])

    return 1


In [ ]:
save_halo_sel(sel, vmax_sel=True, Nhalos=1000)

10 Halos

In [ ]:
sel_10 = {}
sel_10['Custom'] = splitter.subhalo_sel(mhalo_edges=mass_edges, vmax_sel=False, Nhalos=c_Nhalos_10)
sel_10['vmax_Custom'] = splitter.subhalo_sel(mhalo_edges=mass_edges, vmax_sel=True, Nhalos=c_Nhalos_10)

### Plot their mass and concentration

In [ ]:
gal_sel = sel['2']['gal_sel']
parent_mass = sim.sub['parent_halo']['mfof'][gal_sel] * 1e10
halo_vmax = sim.sub['vmax'][sim.fof['halo_firstsub']]
m200b = sim.fof['halo_m200b']
r200b = sim.fof['halo_r200b']
G_newton = 4.3009172706e-9 #Mpc/M_sun * (km/s)**2
halo_v200 = np.sqrt(G_newton*m200b/r200b)
vmax_v200 = halo_vmax / halo_v200

parent_vmax = halo_vmax[sim.sub['fof_index'][gal_sel]]



In [ ]:
fig, ax = plt.subplots(1,2,dpi=150, figsize=(11,5), sharey=True, sharex=True)

ax[0].set_xscale('log')
ax[0].set_yscale('log')

ax[0].set_xlabel('$M_{fof}$')
ax[0].set_ylabel('$V_{max}/V_{200}$')

ax[0].plot(1e10*sim.fof['halo_mfof'][sel['2']['h_idx']], vmax_v200[sel['2']['h_idx']], marker='o', ls='', color='k', ms=1)

for m in range(len(sel_vmax['h_idx'])):
    for v in range(len(sel_vmax['h_idx'][m])):
        ax[0].plot(1e10*sim.fof['halo_mfof'][sel_vmax['h_idx'][m][v]], vmax_v200[sel_vmax['h_idx'][m][v]], marker='o', ls='', color='C'+str(v), ms=1)

ax[0].legend(loc='upper left')

ax[1].set_xscale('log')
ax[1].set_yscale('log')

ax[1].set_xlabel('$M_{fof}$')

ax[1].plot(1e10*sim.fof['halo_mfof'][sel['Custom']['h_idx']], vmax_v200[sel['Custom']['h_idx']], marker='o', ls='', color='C0', ms=1)
for m in range(len(sel['vmax_Custom']['h_idx'])):
    for v in range(len(sel['vmax_Custom']['h_idx'][m])):
        ax[1].plot(1e10*sim.fof['halo_mfof'][sel['vmax_Custom']['h_idx'][m][v]], vmax_v200[sel['vmax_Custom']['h_idx'][m][v]], marker='o', ls='', color=(v/len(sel['vmax_Custom']['h_idx'][m]),0,0), ms=1)

ax[1].legend(loc='upper left')

In [ ]:
fig, ax = plt.subplots(dpi=150, figsize=(5.5,5))

# First axis, plotting halo mass at each bin
ax.set_yscale('log')

ax.set_xlabel('$\log_{10}(M_{\\mathrm{vir}}/(M_{\\odot}/h))$')
ax.set_ylabel("$M_{bin}/M_T$")

m_centers = 0.5*(mass_edges[:,0] + mass_edges[:,1])
ax.plot(m_centers, sel['2']['mh_bin']/sel['2']['mh_tot'], marker='o', ls='', color='C0')
ax.plot(m_centers, utils.dict2d_sum(sel_vmax['mh_bin']) /sel['2']['mh_tot'], marker='^', ls='', color='C1')
ax.plot(m_centers, utils.dict2d_sum(sel['vmax_Custom']['mh_bin']) /sel['2']['mh_tot'], marker='^', ls='', color='C2')
ax.plot(m_centers, sel['Custom']['mh_bin']/sel['2']['mh_tot'], marker='o', ls='', color='C3')
ax.axhline(1, color='k', ls='--')
ax.annotate("$M={:.2f}\%$".format(100*sel['2']['mh_tot']/sel_total['mh_tot']), (12,4e-3), color='C0')
ax.annotate("$M={:.2f}\%$".format(100*sel['vmax_Custom']['mh_tot']/sel_total['mh_tot']), (12.5,0.3), color='C2')
ax.annotate("$M={:.2f}\%$".format(100*sel_vmax['mh_tot']/sel_total['mh_tot']), (11.8,2e-4), color='C1')
ax.annotate("$M={:.2f}\%$".format(100*sel['Custom']['mh_tot']/sel_total['mh_tot']), (12,0.2), color='C3')

# Second axis
ax2 = ax.twinx()
ax2.set_yscale('log')

ax2.plot(m_centers, sel['2']['h_frac'], color='C0')
ax2.plot(m_centers, sel['Custom']['h_frac'], color='C3')
ax2.set_ylabel('$N_{\\mathrm{sel}}/N_{T}$')

### Compute bias for galaxies in each halo-mass bin

In [ ]:
bins = 10**np.array([9,9.4,9.8,10.2,10.6,11,11.5,12])

In [ ]:
bias_total = splitter.bias_sm(gal_sel=sel_total['gal_sel'], sel_mask=sel_total, recompute=False, bins=bins)

In [ ]:
bias = {}

for ns in range(len(sels)):
    bias[sels[ns]] = splitter.bias_sm(gal_sel=sel[sels[ns]]['gal_sel'], sel_mask=sel[sels[ns]], recompute=False, bins=bins)

In [ ]:
bias['vmax_Custom'] = splitter.bias_sm(gal_sel=sel['vmax_Custom']['gal_sel'], sel_mask=sel['vmax_Custom'], recompute=False, bins=bins, vmax_sel=True)

In [ ]:
bias_vmax = splitter.bias_sm(gal_sel=sel_vmax['gal_sel'], sel_mask=sel_vmax, recompute=False, bins=bins, vmax_sel=True)

In [ ]:
fig, ax = plt.subplots(1, 3, dpi=100, figsize=(19,5))

ax[0].set_xscale('log')

ax[0].plot(bias_total['mstar'], bias_total['bias'][:,0], color='C0', alpha=0.8, label='Total')

for ns in range(len(sels)):
    ax[0].plot(bias[sels[ns]]['mstar'], bias[sels[ns]]['bias'][:,0], ls='-', color='C'+str(ns+1), marker='o', ms=4,\
               label='$N_h=${:d}, $M={:.2f}\%$'.format(sel[sels[ns]]['Nh'], 100*sel[sels[ns]]['mh_tot']/sel_total['mh_tot']))
ax[0].plot(bias['vmax_Custom']['mstar'], bias['vmax_Custom']['bias'][:,0], ls='-', color='C3', marker='o', ms=4,\
               label='$V_{max}$'+', $N_h=${:d}, $M={:.2f}\%$'.format(sel['vmax_Custom']['Nh'], 100*sel['vmax_Custom']['mh_tot']/sel_total['mh_tot']))
ax[0].plot(bias_vmax['mstar'], bias_vmax['bias'][:,0], ls='-', color='C4', marker='o', ms=4,\
               label='$V_{max}$'+', $N_h=${:d}, $M={:.2f}\%$'.format(sel_vmax['Nh'], 100*sel_vmax['mh_tot']/sel_total['mh_tot']))

ax[0].set_xlabel('$\\log_{10}(M_*/M_\\odot)$', fontsize=16)
ax[0].set_ylabel('$c_{2=2}$', fontsize=16)

ax[0].set_ylim(-0.3,0.1)

ax[0].legend(loc='lower left')

#############################
ax[1].set_xscale('log')

ax[1].plot(bias[sels[ns]]['mstar'], bias_total['bias'][:,1], color='C0', alpha=0.8, label='Total')

for ns in range(len(sels)):
    ax[1].plot(bias[sels[ns]]['mstar'], bias[sels[ns]]['bias'][:,1], ls='-', color='C'+str(ns+1), marker='o', ms=4,\
               label='$N_h=${:d}, $M={:.2f}\%$'.format(sel[sels[ns]]['Nh'], 100*sel[sels[ns]]['mh_tot']/sel_total['mh_tot']))

ax[1].plot(bias['vmax_Custom']['mstar'], bias['vmax_Custom']['bias'][:,1], ls='-', color='C3', marker='o', ms=4,\
               label='$V_{max}$'+', $N_h=${:d}, $M={:.2f}\%$'.format(sel[sels[ns]]['Nh'], 100*sel[sels[ns]]['mh_tot']/sel_total['mh_tot']))
ax[1].plot(bias_vmax['mstar'], bias_vmax['bias'][:,1], ls='-', color='C4', marker='o', ms=4,\
               label='$V_{max}$'+', $N_h=${:d}, $M={:.2f}\%$'.format(sel_vmax['Nh'], 100*sel_vmax['mh_tot']/sel_total['mh_tot']))

ax[1].set_xlabel('$\\log_{10}(M_*/M_\\odot)$', fontsize=16)
ax[1].set_ylabel('$c_{22=2}$', fontsize=16)

#############################
ax[2].set_xscale('log')

ax[2].plot(bias[sels[ns]]['mstar'], bias_total['bias'][:,2], color='C0', alpha=0.8, label='Total')

for ns in range(len(sels)):
    ax[2].plot(bias[sels[ns]]['mstar'], bias[sels[ns]]['bias'][:,2], ls='-', color='C'+str(ns+1), marker='o', ms=4,\
               label='$N_h=${:d}, $M={:.2f}\%$'.format(sel[sels[ns]]['Nh'], 100*sel[sels[ns]]['mh_tot']/sel_total['mh_tot']))

ax[2].plot(bias['vmax_Custom']['mstar'], bias['vmax_Custom']['bias'][:,2], ls='-', color='C3', marker='o', ms=4,\
               label='$V_{max}$'+', $N_h=${:d}, $M={:.2f}\%$'.format(sel[sels[ns]]['Nh'], 100*sel[sels[ns]]['mh_tot']/sel_total['mh_tot']))
ax[2].plot(bias_vmax['mstar'], bias_vmax['bias'][:,2], ls='-', color='C4', marker='o', ms=4,\
               label='$V_{max}$'+', $N_h=${:d}, $M={:.2f}\%$'.format(sel_vmax['Nh'], 100*sel_vmax['mh_tot']/sel_total['mh_tot']))

ax[2].set_xlabel('$\\log_{10}(M_*/M_\\odot)$', fontsize=16)
ax[2].set_ylabel('$c_{2-2-2-}$', fontsize=16)

In [ ]:
smf['vmax'] = splitter.stellar_mf(gal_sel=sel_vmax['gal_sel'], sel_mask=sel_vmax, nbins=nbins, vmax_sel=True)

## Get the Stellar-Mass Functions

In [ ]:
nbins=20
smf_total = splitter.stellar_mf(gal_sel=sel_total['gal_sel'], sel_mask=sel_total, nbins=nbins)

smf = {}
smf['2'] = splitter.stellar_mf(gal_sel=sel['2']['gal_sel'], sel_mask=sel['2'], nbins=nbins)
smf['vmax'] = splitter.stellar_mf(gal_sel=sel_vmax['gal_sel'], sel_mask=sel_vmax, nbins=nbins, vmax_sel=True)
smf['Custom'] = splitter.stellar_mf(gal_sel=sel['Custom']['gal_sel'], sel_mask=sel['Custom'], nbins=nbins)
smf['vmax_Custom'] = splitter.stellar_mf(gal_sel=sel['vmax_Custom']['gal_sel'], sel_mask=sel['vmax_Custom'], nbins=nbins, vmax_sel=True)

In [ ]:
GAMA = np.array([
    [6.875, -0.691, 0.176],
    [7.125, -1.084, 0.125],
    [7.375, -1.011, 0.071],
    [7.625, -1.349, 0.092],
    [7.875, -1.287, 0.079],
    [8.125, -1.544, 0.071],
    [8.375, -1.669, 0.045],
    [8.625, -1.688, 0.032],
    [9.125, -1.886, 0.020],
    [9.375, -2.055, 0.014],
    [9.625, -2.142, 0.010],
    [9.875, -2.219, 0.009],
    [10.125, -2.274, 0.009],
    [10.375, -2.292, 0.009],
    [10.625, -2.361, 0.010],
    [10.875, -2.561, 0.013],
    [11.125, -2.922, 0.019],
    [11.375, -3.414, 0.032],
    [11.625, -4.704, 0.138]
])



In [ ]:
GAMA_corr = np.zeros((GAMA.shape[0], 4))
GAMA_corr[:,0] = np.log10( 10**GAMA[:,0] / 0.7**2 )
GAMA_corr[:,1] = np.log10( 10**GAMA[:,1] * 0.7**3 )

# Transforming the errors is a bit more involved
xn = 10**GAMA[:,1] * ( 1 - 10**(-GAMA[:,2]) )
xp = 10**GAMA[:,1] * ( 10**(GAMA[:,2]) - 1 )

GAMA_corr[:,2] = GAMA[:,1] - np.log10( 10**GAMA[:,1] - xn  )
GAMA_corr[:,3] = np.log10( 10**GAMA[:,1] + xp  ) - GAMA[:,1]

In [ ]:
fig = plt.figure(dpi=200, figsize=(5.5,5))

ax1=fig.add_axes((.1,.3,.8,.6))

# centers = 0.5*(smf_total['bins'][1:] + smf_total['bins'][:-1])
# bins = smf['2']['bins']

ax1.plot(np.log10(smf['2']['mstar']), np.log10(smf['2']['smf']), label='$N_h=40$')
ax1.plot(np.log10(smf['vmax']['mstar']), np.log10(smf['vmax']['smf']), label='$N_h=40$, $V_{max}$')
ax1.plot(np.log10(smf['Custom']['mstar']), np.log10(smf['Custom']['smf']), label='$N_h=1000$')
ax1.plot(np.log10(smf['vmax_Custom']['mstar']), np.log10(smf['vmax_Custom']['smf']), label='$N_h=1000$, $V_{max}$')
ax1.plot(np.log10(smf_total['mstar']), np.log10(smf_total['smf']), label='Total', color='k')


ax1.errorbar(GAMA_corr[:,0], GAMA_corr[:,1], yerr=[GAMA_corr[:,2], GAMA_corr[:,3]], color='r', marker='o', capsize=3, ms=4, fillstyle='none', ls='')

ax1.legend(loc='lower center')

ax1.set_xlim(9, 13) 
ax1.set_ylim(-6, -2) 

ax1.set_ylabel('$\mathrm{d}n(M)/\mathrm{dlog}_{10}M_*[\mathrm{Mpc}]^{-3}$')
ax1.set_xticklabels([])

ax2=fig.add_axes((.1,.1,.8,.2))        

ax2.plot(GAMA_corr[:,0], np.interp( GAMA_corr[:,0], np.log10(smf['2']['mstar']), np.log10(smf['2']['smf']) )-GAMA_corr[:,1] )
ax2.plot(GAMA_corr[:,0], np.interp( GAMA_corr[:,0], np.log10(smf['vmax']['mstar']), np.log10(smf['vmax']['smf']) )-GAMA_corr[:,1] )
ax2.plot(GAMA_corr[:,0], np.interp( GAMA_corr[:,0], np.log10(smf['Custom']['mstar']), np.log10(smf['Custom']['smf']) )-GAMA_corr[:,1] )
ax2.plot(GAMA_corr[:,0], np.interp( GAMA_corr[:,0], np.log10(smf['vmax_Custom']['mstar']), np.log10(smf['vmax_Custom']['smf']) )-GAMA_corr[:,1] )
ax2.plot(GAMA_corr[:,0], np.interp( GAMA_corr[:,0], np.log10(smf_total['mstar']), np.log10(smf_total['smf']) )-GAMA_corr[:,1], color='k')

ax2.errorbar(GAMA_corr[:,0], np.zeros(len(GAMA_corr[:,0])), yerr=[GAMA_corr[:,2], GAMA_corr[:,3]], marker='o', color='r', capsize=3, ms=4, fillstyle='none', ls='')

ax2.set_xlim(9, 13) 
ax2.set_ylim(-0.5,0.5)

ax2.set_yticks([-0.5,-0.4,-0.3,-0.2,-0.1,0.0,0.1,0.2,0.3,0.4,0.5], minor=True)

#ax2.fill_between(centers, -0.05,0.05, color='gray', alpha=0.1)
#ax2.fill_between(centers, -0.1,0.1, color='gray', alpha=0.1)

ax2.set_ylabel('$\log_{10}(\mathrm{Sim}/GAMA)$')
ax2.set_xlabel('Stellar Mass $M_*[M_\odot]$')


In [ ]:
sel = np.where( (sim.sub['LenType'][:,4]>200) & (np.sum(sim.sub['MassType'], axis=1)>1) )

U__r = sim.sub['StellarPhotometrics'][sel,0] - sim.sub['StellarPhotometrics'][sel,5]
M_r = sim.sub['StellarPhotometrics'][sel,5]

g__i = sim.sub['StellarPhotometrics'][sel,4] - sim.sub['StellarPhotometrics'][sel,6]
M_i = sim.sub['StellarPhotometrics'][sel,6]

In [ ]:
fig, ax = plt.subplots(1, 2, dpi=150, figsize=(11,5))

ax[0].scatter(M_r[::10000], U__r[::10000], s=0.01, alpha=0.02)
ax[1].scatter(M_i[::10000], g__i[::10000], s=0.01, alpha=0.02)

ax[0].axhline(1.35, color='k', lw=0.9)

# Setting labels
ax[0].set_xlabel('$M_r$')
ax[0].set_ylabel('$u-r$')

ax[1].set_xlabel('$M_i$')
ax[1].set_ylabel('$g-i$')

ax[0].set_xlim(-18,-26)
ax[1].set_xlim(-18,-26)

### Split these subpopulations

In [ ]:
sel_t = {}
sel_t['spi'] = copy.deepcopy(sel_total)
sel_t['ell'] = copy.deepcopy(sel_total)

sSFR = 1e9 * sim.sub['SFR'] / (1e10 * sim.sub['MassType'][:, 4])

for m in range(len(sel_total['sel'])):
    sel_t['spi']['sel'][m] = sel_t['spi']['sel'][m][np.where( (sSFR[sel_t['spi']['sel'][m]] > 0.1) ) ]
    sel_t['ell']['sel'][m] = sel_t['ell']['sel'][m][np.where( (sSFR[sel_t['ell']['sel'][m]] < 0.1) ) ]

In [ ]:
bias_t = {}

bias_t['spi'] = splitter.bias_sm(sel_mask=sel_t['spi'], recompute=False)
bias_t['ell'] = splitter.bias_sm(sel_mask=sel_t['ell'], recompute=False)

In [ ]:
fig, ax = plt.subplots(dpi=100, figsize=(5.5,5))
ax.fill_between(ms_centers, 0.9*bias_t['spi']['bias'][:,1], 1.1*bias_t['spi']['bias'][:,1], color='C0', alpha=0.8, label='$sSFR > 0.1$')
ax.fill_between(ms_centers, 0.9*bias_t['ell']['bias'][:,1], 1.1*bias_t['ell']['bias'][:,1], color='C3', alpha=0.8, label='$sSFR < 0.1$')

ax.legend(loc='lower left')
ax.set_ylabel('$c_{J_{2=2}}$')
ax.set_xlabel('$M_*[M_{\odot}/h]$')

In [ ]:
sel_t = {}
sel_t['red'] = copy.deepcopy(sel_total)
sel_t['blue'] = copy.deepcopy(sel_total)

U__r = sim.sub['StellarPhotometrics'][:,0] - sim.sub['StellarPhotometrics'][:,5]

for m in range(len(sel_total['sel'])):
    sel_t['red']['sel'][m] = sel_t['red']['sel'][m][np.where( (U__r[sel_t['red']['sel'][m]] > 1.35) ) ]
    sel_t['blue']['sel'][m] = sel_t['blue']['sel'][m][np.where( (U__r[sel_t['blue']['sel'][m]] < 1.35) ) ]

In [ ]:
bias_t = {}

bias_t['red'] = splitter.bias_sm(sel_mask=sel_t['red'], recompute=False)
bias_t['blue'] = splitter.bias_sm(sel_mask=sel_t['blue'], recompute=False)

In [ ]:
fig, ax = plt.subplots(dpi=100, figsize=(5.5,5))
ax.fill_between(ms_centers, 0.9*bias_t['red']['bias'][:,1], 1.1*bias_t['red']['bias'][:,1], color='C3', alpha=0.8, label='Total')
ax.fill_between(ms_centers, 0.9*bias_t['blue']['bias'][:,1], 1.1*bias_t['blue']['bias'][:,1], color='C0', alpha=0.8, label='Total')


## What about gas-fraction

In [ ]:
gas_prop = {}

m500_edges = np.array([[12,12.5],[12.5,13],[13,13.5],[13.5,14],[14,14.5],[14.5,15]])

gas_prop['total'] = splitter.gas_frac(m500_edges=m500_edges, vmax_sel=False, sel_mask=sel_total)
gas_prop['2'] = splitter.gas_frac(m500_edges=m500_edges, vmax_sel=False, sel_mask=sel['2'])
gas_prop['Custom'] = splitter.gas_frac(m500_edges=m500_edges, vmax_sel=False, sel_mask=sel['Custom'])
gas_prop['vmax_2'] = splitter.gas_frac(m500_edges=m500_edges, vmax_sel=True, sel_mask=sel_vmax)
gas_prop['vmax_Custom'] = splitter.gas_frac(m500_edges=m500_edges, vmax_sel=True, sel_mask=sel['vmax_Custom'])

In [ ]:
fig = plt.figure(dpi=200, figsize=(5,5.5))

ax1=fig.add_axes((.1,.3,.8,.6))

ax1.set_xscale('log')

ax1.plot(gas_prop['total']['m500c'], gas_prop['total']['f_gas'],lw=0.8, color='k', label='Total', ms=3)
ax1.plot(gas_prop['2']['m500c'], gas_prop['2']['f_gas'], marker='o', ls='-', lw=0.8, label='$N_h=40$', ms=3)
ax1.plot(gas_prop['vmax_2']['m500c'], gas_prop['vmax_2']['f_gas'], marker='o', ls='-', lw=0.8, label='$N_h=40$, $V_{max}$', ms=3)
ax1.plot(gas_prop['Custom']['m500c'], gas_prop['Custom']['f_gas'], marker='o', ls='-', lw=0.8, label='$N_h=1000$', ms=3)
ax1.plot(gas_prop['vmax_Custom']['m500c'], gas_prop['vmax_Custom']['f_gas'], marker='o', ls='-', lw=0.8, label='$N_h=1000$, $V_{max}$', ms=3)

ax1.legend(fontsize=6)

ax1.set_ylabel('$f_{gas}$')
ax1.set_xlabel('$M_{500c}[M_{\odot}/h]$')

In [ ]:
smf_total = {}; smf = {}; mhalos = {}; Nhalos = {}
for m in range(mass_edges.shape[0]):
    mhalos[m] = {}
    Nhalos[m] = {}

    smf_total[m], mhalos[m]['total'], _ = stellar_mf(sim=sim, mass_edges=mass_edges[m], nbins=Nbins)
    
    smf[m] = {}

    Nhalos[m]['10'] = max( min(400, int(0.1*nhalos[m]['total']) ), 15 )
    smf[m]['10'], mhalos[m]['10'], _ = stellar_mf(sim=sim, mass_edges=mass_edges[m], Nhalos=Nhalos[m]['10'], nbins=Nbins)
    Nhalos[m]['1'] = max( min(400, int(0.01*nhalos[m]['total']) ), 15 )
    smf[m]['1'], mhalos[m]['1'], _ = stellar_mf(sim=sim, mass_edges=mass_edges[m], Nhalos=Nhalos[m]['1'], nbins=Nbins)
    Nhalos[m]['0.5'] = max( min(400, int(0.005*nhalos[m]['total']) ), 15 )
    smf[m]['0.5'], mhalos[m]['0.5'], _ = stellar_mf(sim=sim, mass_edges=mass_edges[m], Nhalos=Nhalos[m]['0.5'], nbins=Nbins)


In [ ]:
for m in range(mass_edges.shape[0]):
    fig = plt.figure(dpi=200, figsize=(5.5,5))

    ax1=fig.add_axes((.1,.3,.8,.6))

    ax1.set_xscale('log')

    ax1.hist(0.5*(smf_total[m][1][:-1] + smf_total[m][1][1:]),\
             bins=smf_total[m][1], weights=smf_total[m][0]/np.sum(smf_total[m][0]), log=True, histtype='step', color='k', label='Full');

    ax1.hist(0.5*(smf[m]['10'][1][:-1] + smf[m]['10'][1][1:]),\
             bins=smf[m]['10'][1], weights=smf[m]['10'][0]/np.sum(smf[m]['10'][0]), log=True, histtype='step',
             label='$N_h={:d}$, ${:.2f}\%$'.format(Nhalos[m]['10'], 100*Nhalos[m]['10']/nhalos[m]['total']), color='C0');

    ax1.hist(0.5*(smf[m]['1'][1][:-1] + smf[m]['1'][1][1:]),\
             bins=smf[m]['1'][1], weights=smf[m]['1'][0]/np.sum(smf[m]['1'][0]), log=True, histtype='step',
             label='$N_h={:d}$, ${:.2f}\%$'.format(Nhalos[m]['1'], 100*Nhalos[m]['1']/nhalos[m]['total']), color='C1');

    ax1.hist(0.5*(smf[m]['0.5'][1][:-1] + smf[m]['0.5'][1][1:]),\
             bins=smf[m]['0.5'][1], weights=smf[m]['0.5'][0]/np.sum(smf[m]['0.5'][0]), log=True, histtype='step',
             label='$N_h={:d}$, ${:.2f}\%$'.format(Nhalos[m]['0.5'], 100*Nhalos[m]['0.5']/nhalos[m]['total']), color='C2');

    ax1.legend(loc='lower center')

    ax1.set_ylabel('$\\bar{n}$ (Arbitrary Units)')

    ax1.set_title('$\log(M_h/M_\odot) \in [{:.1f},{:.1f}]$'.format(mass_edges[m][0], mass_edges[m][1]))

    ax2=fig.add_axes((.1,.1,.8,.2))        

    ax2.set_xscale('log')

    centers = 0.5*(smf[m]['0.5'][1][:-1] + smf[m]['0.5'][1][1:])

    ax2.plot( centers, (smf_total[m][0]/np.sum(smf_total[m][0])) / (smf[m]['10'][0]/np.sum(smf[m]['10'][0])) - 1, label='$N_h=10^4$', color='C0')
    ax2.plot( centers, (smf_total[m][0]/np.sum(smf_total[m][0])) / (smf[m]['1'][0]/np.sum(smf[m]['1'][0])) - 1, label='$N_h=10^3$', color='C1')
    ax2.plot( centers, (smf_total[m][0]/np.sum(smf_total[m][0])) / (smf[m]['0.5'][0]/np.sum(smf[m]['0.5'][0])) - 1, label='$N_h=200$', color='C2')

    ax2.set_ylim(-0.5,0.5)

    ax2.fill_between(centers, -0.1,0.1, color='gray', alpha=0.1)

    ax2.set_ylabel('Deviation')
    ax2.set_xlabel('$M_*[M_{\odot}/h]$')

In [ ]:
smf_total_sum = np.zeros(len(smf_total[0][0]))
smf_sum = {}
smf_sum['10'] = np.zeros(len(smf_total[0][0]))
smf_sum['1'] = np.zeros(len(smf_total[0][0]))
smf_sum['0.5'] = np.zeros(len(smf_total[0][0]))


m_sum ={}
m_sum['total'] = 0
m_sum['10'] = 0
m_sum['1'] = 0
m_sum['0.5'] = 0
for m in range(12):
    
    smf_total_sum += smf_total[m][0]
    smf_sum['10'] += smf[m]['10'][0] / (np.sum(smf[m]['10'][0])/np.sum(smf_total[m][0]))
    smf_sum['1'] += smf[m]['1'][0] / (np.sum(smf[m]['1'][0])/np.sum(smf_total[m][0]))
    smf_sum['0.5'] += smf[m]['0.5'][0] / (np.sum(smf[m]['0.5'][0])/np.sum(smf_total[m][0]))

    m_sum['total'] += np.sum(mhalos[m]['total'])
    m_sum['10'] += np.sum(mhalos[m]['10'])
    m_sum['1'] += np.sum(mhalos[m]['1'])
    m_sum['0.5'] += np.sum(mhalos[m]['0.5'])

In [ ]:
fig = plt.figure(dpi=200, figsize=(5.5,5))

ax1=fig.add_axes((.1,.3,.8,.6))

ax1.set_xscale('log')

centers = 0.5*(smf_total[0][1][:-1] + smf_total[0][1][1:])
bins = smf_total[0][1]

ax1.hist(centers, bins=bins, weights=3.5*smf_full[0]/np.sum(smf_full[0]),\
         log=True, histtype='step', color='k', label='No Mass Cut', ls='--');

ax1.hist(centers, bins=bins, weights=smf_total_sum/np.sum(smf_total_sum),\
         log=True, histtype='step', color='k', label='Full');

ax1.hist(centers, bins=bins, weights=smf_sum['10']/np.sum(smf_sum['10']), log=True, histtype='step',\
            label='$N_h=$, $M={:.2f}\%$'.format(100*m_sum['10']/m_sum['total']), color='C0');

ax1.hist(centers, bins=bins, weights=smf_sum['1']/np.sum(smf_sum['1']), log=True, histtype='step',\
         label='$N_h=$, $M={:.2f}\%$'.format(100*m_sum['1']/m_sum['total']), color='C1');

ax1.hist(centers, bins=bins, weights=smf_sum['0.5']/np.sum(smf_sum['0.5']), log=True, histtype='step',\
         label='$N_h=$, $M={:.2f}\%$'.format(100*m_sum['0.5']/m_sum['total']), color='C2');

ax1.legend(loc='lower center')

ax1.set_ylabel('$\\bar{n}$ (Arbitrary Units)')

# ax1.set_title('$\log(M_h/M_\odot) \in [{:.1f},{:.1f}]$'.format(mass_edges[m][0], mass_edges[m][1]))

ax2=fig.add_axes((.1,.1,.8,.2))        

ax2.set_xscale('log')

ax2.plot( centers, (smf_total_sum/np.sum(smf_total_sum)) / (smf_sum['10']/np.sum(smf_sum['10'])) - 1,  color='C0')
ax2.plot( centers, (smf_total_sum/np.sum(smf_total_sum)) / (smf_sum['1']/np.sum(smf_sum['1'])) - 1,  color='C1')
ax2.plot( centers, (smf_total_sum/np.sum(smf_total_sum)) / (smf_sum['0.5']/np.sum(smf_sum['0.5'])) - 1, color='C2')

ax2.set_ylim(-0.2,0.2)

ax2.fill_between(centers, -0.05,0.05, color='gray', alpha=0.1)
ax2.fill_between(centers, -0.1,0.1, color='gray', alpha=0.1)

ax2.set_ylabel('Deviation')
ax2.set_xlabel('$M_*[M_{\odot}/h]$')